# 03 — Retrieval / curate() Strategy Benchmark

`curate()` is the Librarian tool — the single retrieval entry point that searches
compiled notes (Tier 2), escalates to raw chunks (Tier 1) when notes are sparse, and
returns a confidence-weighted context. This notebook benchmarks `curate()` itself,
against a real isolated vault built from a real corpus slice, with real embeddings:

1. **Pure cosine vs. hybrid RRF** (`curate.use_hybrid_retrieval`) — does fusing BM25
   lexical search with cosine similarity change what gets retrieved?
2. **Escalation behavior** — when does `curate()` fall back from notes to chunks
   (`escalation_min_notes`, `escalation_max_distance`)?
3. **Confidence weighting** — how much do `reviewed` vs. `unreviewed` notes each
   contribute to the final confidence score?
4. **RRF fusion mechanics** — for one query, the actual cosine rank / BM25 rank /
   fused rank of every candidate note.

No mock data anywhere: real Ollama embeddings, a real ingested + segmented + note-
converted vault, and the production `curate()` function unmodified.

In [ ]:
import sys
import tempfile
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.config import load_settings
from infrastructure.context import build_context
from notebooks._lib.embedding_cache import probe_provider

settings = load_settings()
ctx = build_context(settings)
probe_provider(ctx)
print("EgoVault Retrieval / curate() Benchmark ready — embedding provider reachable.")

## 1. Build an isolated real vault

Ingest a real 30-page slice of Marcus Aurelius, then convert several note
candidates into notes — alternating `reviewed`/`unreviewed` so confidence
weighting is visible later.

In [ ]:
from infrastructure.db import init_db, init_system_db
from infrastructure.vault_db import VaultDB
from infrastructure.vault_writer import write_note as _write_note
from notebooks._lib.embedding_cache import wrap_ctx_embed
from notebooks.demo_retrieval_curate_benchmark import setup_vault, run_query_both_modes, QUERY_SET

tmp_dir = tempfile.TemporaryDirectory()
tmp_path = Path(tmp_dir.name)
vault_db, sys_db = tmp_path / "vault.db", tmp_path / "system.db"
vault_dir, media_dir = tmp_path / "vault", tmp_path / "media"
vault_dir.mkdir()
media_dir.mkdir()
init_db(vault_db)
init_system_db(sys_db)

ctx.db = VaultDB(vault_db)
ctx.system_db_path = sys_db
ctx.vault_path = vault_dir
ctx.media_path = media_dir
ctx.write_note = _write_note
wrap_ctx_embed(ctx)

source, notes = setup_vault(ctx)

## 2. Run the fixed query set through both retrieval modes

Three Stoic-themed queries the corpus should answer well, plus one deliberately
out-of-corpus query ("the price of copper on the London Metal Exchange") to see
how retrieval behaves when there's nothing relevant to find.

In [ ]:
query_results = {}
for q in QUERY_SET:
    print(f'Running: "{q}"')
    query_results[q] = run_query_both_modes(q, ctx)

for q, (cosine_r, hybrid_r) in query_results.items():
    n_notes_c = sum(1 for s in cosine_r.sources if s.tier == "note")
    n_chunks_c = sum(1 for s in cosine_r.sources if s.tier == "chunk")
    n_notes_h = sum(1 for s in hybrid_r.sources if s.tier == "note")
    n_chunks_h = sum(1 for s in hybrid_r.sources if s.tier == "chunk")
    print(f'"{q}"')
    print(f"    cosine -> confidence={cosine_r.confidence}, notes={n_notes_c}, chunks={n_chunks_c}")
    print(f"    hybrid -> confidence={hybrid_r.confidence}, notes={n_notes_h}, chunks={n_chunks_h}")

## 3. Confidence + tier composition, cosine vs. hybrid

Watch the last query (out-of-corpus): hybrid RRF surfaces fewer confident notes,
which lets chunk-tier escalation actually show up in the final result — pure
cosine mode fills all 5 result slots with weakly-relevant notes and never gets
to the chunks.

In [ ]:
from notebooks.demo_retrieval_curate_benchmark import plot_mode_comparison

assets_dir = PROJECT_ROOT / "notebooks" / "assets"
assets_dir.mkdir(parents=True, exist_ok=True)
plot_mode_comparison(query_results, assets_dir / "curate_mode_comparison.png")

from IPython.display import Image, display
display(Image(filename=str(assets_dir / "curate_mode_comparison.png")))

## 4. Confidence weighting breakdown

`curate()`'s confidence score is a weighted average: `reviewed_note_weight=1.0`,
`unreviewed_note_weight=0.7` (from `config/system.yaml`). This shows exactly how
much each retrieved source contributes.

In [ ]:
from notebooks.demo_retrieval_curate_benchmark import plot_confidence_weighting

rich_query = QUERY_SET[0]
plot_confidence_weighting(rich_query, query_results[rich_query][0], ctx, assets_dir / "curate_confidence_weighting.png")
display(Image(filename=str(assets_dir / "curate_confidence_weighting.png")))

## 5. RRF fusion mechanics

The raw mechanics behind hybrid mode: cosine rank, BM25 (FTS5) rank, and the
Reciprocal Rank Fusion score that combines them (`infrastructure.db._rrf_fuse`,
`rrf_k` from config).

In [ ]:
from notebooks.demo_retrieval_curate_benchmark import plot_rrf_fusion

plot_rrf_fusion(rich_query, ctx, assets_dir / "curate_rrf_fusion.png")
display(Image(filename=str(assets_dir / "curate_rrf_fusion.png")))
tmp_dir.cleanup()

## 6. Takeaway

`curate()`'s escalation-to-chunks path is real but easy to miss: because sources
are `(notes + chunks)[:limit]` and notes are listed first, escalation only becomes
visible in the final result when there are *fewer* note matches than the limit —
otherwise chunks are computed internally and then silently truncated away. Hybrid
RRF changes which notes rank highly enough to fill those slots, which is why the
out-of-corpus query behaves differently between modes even though both modes
technically "escalate" internally. This is a real engine behavior worth knowing
before relying on chunk-tier evidence always showing up when notes are sparse.